🌐 [English](getting_started.ipynb) | [한국어](getting_started.ko.ipynb) | **日本語** | [中文](getting_started.zh.ipynb)

# DEEPX Tutorial 01 - DEEPX SDK のインストール方法
この最初のチュートリアルでは、DEEPX SDK をインストールし、セットアップが正常に完了したことを確認する方法を説明します。環境の準備、SDK のインストール、そして DEEPX NPU デバイス(DX-M1、DX-M1M、DX-H1 Quattro)がシステムで正しく認識されていることの確認までを学びます。

## 学習目標

このチュートリアルを終えると、DX-All Suite を正しくインストールし、DEEPX NPU デバイス上で基本的なフローを実行できるようになります。

## 前提条件

**注:** 以下の要件はこのチュートリアルのためのものであり、**DEEPX 製品を使用するための必須条件ではありません。**
- OS: Linux (Ubuntu 20.04/22.04/24.04/26.04, Debian 12/13)
- RAM: 8G (DX-Compiler を使う場合は 16G)
- ストレージ: 40 GB 以上
- DEEPX NPU: DX-M1、DX-M1M、DX-H1 Quattro
- CPU: x86_64 では DX-Compiler + DX-Runtime、aarch64 では DX-Runtime のみ
- ダウンロード: DX-All Suite のクローン(約 600 MB)、DX-COM インストーラー(数百 MB)、3.3 節のサンプルモデル 24 個と共有サンプル動画(それぞれ約 1.2 GB)
- 所要時間: 30〜60 分。3.1 の DX-Runtime ビルドだけで 10〜30 分かかり、その後に再起動が 1 回必要です
- `sudo`: 両方のインストーラーで必要です。各インストール手順はまずセル内でパスワードなしの `sudo` が使えるかを確認し、使えない場合はターミナル用のコマンドを表示します

このチュートリアルと以降のチュートリアルのターミナルコマンドでは、DX-All Suite のディレクトリを `<DX_ALL_SUITE_DIR>` と表記します。既定値は `~/dx-all-suite` で、1.1 節がこのホストに設定された値を表示します。

## DXNN® - DEEPX NPU SDK の紹介 (DX-AS: DX-All Suite)

DX-AS(DX-All Suite)は、DEEPX デバイスを使って AI モデルのコンパイルと推論を行うためのフレームワークとツールの統合環境です。個別のツールをインストールして統合環境を構築することもできますが、DX-AS は各ツールのバージョンを揃えることで最適な互換性を維持します。

![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/resources/dxnn_sdk_illustration.png)

DEEPX SDK は大きく 2 つの部分に分かれています。

1 つ目は AI モデルコンパイル環境で、AI モデルを DEEPX NPU 上で効率的に実行できる最適化された形式に変換します。

2 つ目は AI モデルランタイム環境で、コンパイル済みの AI モデルを実際の DEEPX NPU ハードウェア上で実行して結果を生成します。

DX-All Suite を使えば、両方のコンポーネントを個別に管理することなく、まとめてセットアップできます。

![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/img/dx-as.png)


![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/img/DXNN-SDK-Simple-Architecture.png)



理解を助けるために、DX-SDK に関する YouTube 動画が 2 本あります。
- [Youtube - DEEPX SDK Introduction](https://www.youtube.com/watch?v=Js6Soex0WI4) | ダウンロード: [EN](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_ENG.mp4) · [中文](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_CH.mp4) · [한국어](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_KO.mp4)

<img src="assets/youtube-dx-sdk.png" style="max-width: 1000px;">

## 1. DX-All Suite のダウンロード

### 1.1 インストールの設定

チュートリアルは次の順序で DX-All Suite を探します。

1. 環境変数 `DX_ALL_SUITE_DIR`、
2. `dx-tutorials/config.json`(この節で作成され、git では追跡されません)、
3. `~/dx-all-suite` や `dx-tutorials` の隣にある `dx-all-suite` フォルダーなど、一般的な場所の自動検出、
4. 既定の場所 `~/dx-all-suite`。

次のセルは、見つかった場所とその値の出所を表示します。SDK がまだインストールされていない場合、既定の場所は `MISSING` と表示されます。クローン手順の前では正常な状態です。

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial, save_tutorial_config

# Modules used by cells later in this notebook.
import shlex, shutil, subprocess
from IPython.display import Markdown, display

paths = setup_tutorial()


#### 1.1.1 インストールディレクトリ

上に表示された場所をそのまま使う場合は、次のセルを変更しないでください。DX-All Suite を別の場所にインストールしたい場合(またはすでにインストール済みの場合)にのみ `DX_ALL_SUITE_DIR` を編集します。セルを実行すると選択内容が `dx-tutorials/config.json` に保存され、他のすべてのチュートリアルが同じ場所を使います。

ターミナルからいつでも設定することもできます(`dx-tutorials` ディレクトリで実行)。

```bash
python tutorial_paths.py --set ~/my/dx-all-suite
```

In [ ]:
####################################################################
# Keep the detected location, or edit this value to choose another one.
# Example: DX_ALL_SUITE_DIR = Path("~/my/dx-all-suite")
####################################################################
DX_ALL_SUITE_DIR = paths.dx_all_suite_dir
####################################################################

save_tutorial_config(dx_all_suite_dir=DX_ALL_SUITE_DIR)
paths = setup_tutorial()


#### 1.1.2 Git ブランチ

別の SDK バージョンが必要な場合にのみ、下のブランチ名を変更してください。
現在のブランチ名は `main` です。

In [ ]:
####################################################################
# Edit this value to choose the DX-All Suite Git branch.
# Current default branch name is "main"
####################################################################
DX_ALL_SUITE_BRANCH = "main"
####################################################################

save_tutorial_config(git_branch=DX_ALL_SUITE_BRANCH)
paths = setup_tutorial(quiet=True)

print(f"DX-All Suite Git branch: {DX_ALL_SUITE_BRANCH}")
print(f"Updated configuration:    {paths.config_path}")


### 1.2 DX-All Suite のクローン

次のセルは、設定した場所に **DX-All Suite が存在しない場合にのみ**、以下のコマンドを実行します。ディレクトリとブランチは上で保存した設定から取得され、セル内では `{DX_ALL_SUITE_DIR}` と `{DX_ALL_SUITE_BRANCH}` として表示されます。

```bash
git clone --depth 1 --shallow-submodules --recurse-submodules --progress \
    --branch main https://github.com/DEEPX-AI/dx-all-suite.git <DX_ALL_SUITE_DIR>
```

- `--depth 1 --shallow-submodules` はすべてのリポジトリの最新コミットだけを取得します。数 GB ではなく約 600 MB のダウンロードとディスク上 1.6 GB で済みます。
- `--recurse-submodules` はサブモジュール(`dx-runtime`、`dx-compiler`、`dx-modelzoo` とその内部のリポジトリ)もチェックアウトします。
- `--progress` は出力がターミナルでなくても進捗を表示するので、セル内で進み具合を確認できます。

ネットワークによって 1〜10 分かかります。■ ボタンでセルを停止し、後で再実行できます。完了すると最後の行に `Submodule path ... checked out` と表示されます。

前回のクローンがトップレベルのチェックアウト後に中断された場合、ディレクトリには `.git` がありますが一部のサブモジュールが空です。その場合、セルは再クローンの代わりに次のコマンドを実行します。

```bash
git -C <DX_ALL_SUITE_DIR> submodule update --init --recursive --depth 1 --progress
```

セルを使わずに、ターミナル(**File > New > Terminal**)でどちらかのコマンドを自分で実行しても構いません。下のステータスセルはどちらの場合も同じように動作します。

In [ ]:
# Clone DX-All Suite only when it is missing. The git commands below are the ones to use in a terminal.
if paths.check("sdk")[0].ok:
    print(f"DX-All Suite is already present: {DX_ALL_SUITE_DIR}")
    print("Nothing to clone. Continue with the next cell.")

elif (DX_ALL_SUITE_DIR / ".git").exists():
    print(f"Finishing an interrupted clone: initializing submodules in {DX_ALL_SUITE_DIR}\n")
    !git -C "{DX_ALL_SUITE_DIR}" submodule update --init --recursive --depth 1 --progress

else:
    print(f"Cloning branch {DX_ALL_SUITE_BRANCH} into {DX_ALL_SUITE_DIR}\n")
    !git clone --depth 1 --shallow-submodules --recurse-submodules --progress \
        --branch "{DX_ALL_SUITE_BRANCH}" https://github.com/DEEPX-AI/dx-all-suite.git "{DX_ALL_SUITE_DIR}"

paths = setup_tutorial(quiet=True)
if not paths.check("sdk")[0].ok:
    raise RuntimeError(
        f"DX-All Suite is still not usable at {DX_ALL_SUITE_DIR}. "
        "Read the git message above (network, disk space, or a non-empty directory), fix it, and run this cell again."
    )
print("\nDX-All Suite is ready:", DX_ALL_SUITE_DIR)


### 1.3 現在の状態と次のステップ

次のセルは、これまでにインストールされたものと、`MISSING` と表示された項目についてはそれを提供する正確な手順を表示します。このノートブックに戻ってくるたびに再実行してください。繰り返し実行しても安全です。

In [ ]:
# Current state of the installation, with the next step for anything that is missing.
paths = setup_tutorial(needs=["sdk_branch", "sdk_submodules", "dx_com", "dx_rt"])

print()
print(f"Jupyter Python     : {sys.executable}")
print(f"DX-Compiler Python : {DX_COMPILER_VENV / 'bin' / 'python'}")
print()

missing = paths.missing
if any(result.name == "sdk" for result in missing):
    missing = [result for result in missing if result.name == "sdk"]

if not missing:
    print("Section 1 is complete. Continue with section 2 (DX-Compiler) and section 3 (DX-Runtime).")
else:
    print("Next steps, in order:")
    for number, result in enumerate(missing, 1):
        print(f"  {number}. {result.name}: {result.hint}")


### 1.4 DX-All Suite の中身

クローンには 3 つの Git サブモジュールが含まれています。
 - `dx-compiler` (**DX-Compiler**): ONNX モデルを DXNN に変換します。第 2 節でインストールします。
 - `dx-runtime` (**DX-Runtime**): コンパイル済みモデルを DEEPX NPU 上で実行します。第 3 節でインストールします。
 - `dx-modelzoo`: 後のチュートリアルで使用するコンパイル済みモデルとコンパイラ設定です。

In [ ]:
!tree -L 1 "{DX_ALL_SUITE_DIR}"


DX-Runtime は次を提供します。
 - `DX-APP`: ユーザーアプリケーションの観点から DX NPU の使い方を示す DEEPX アプリケーションテンプレート
 - `DX-FW`: NPU ファームウェアバイナリ
 - `DX-RT`: DX NPU を使った推論タスクを最適化して実行するために設計されたフレームワーク
 - `DX-NPU Driver`: DX NPU 用の Linux カーネルドライバー
 - `DX-STREAM`: DX NPU 向けの GStreamer ベースのビジョン AI アプリケーション開発ツール

In [ ]:
!tree -L 1 "{DX_RUNTIME_DIR}"


## 2. DX-Compiler のインストール

詳細は [DX-All Suite インストールガイド](https://github.com/DEEPX-AI/dx-all-suite/blob/main/docs/source/02_Setting_Up_Environment.md)を参照してください。

DX-Compiler 環境はビルド済みのバイナリを提供し、ソースコードは含みません。インストーラーは各モジュールをリモートサーバーからダウンロードします。

| コマンド | インストール対象 | `sudo` |
|---|---|---|
| `./dx-compiler/install.sh --target=dx_com` | 専用の Python 環境に入る DX-COM コンパイラ(`dxcom`) | すでに存在していても `apt-get install python3-dev python3-venv` を実行 |
| `./dx-compiler/install.sh --target=dx_tron` | DX-TRON モデルビューアー(`.deb` パッケージ) | `apt-get` でパッケージをインストール |
| `./dx-compiler/install.sh` | 両方 | 上記の両方 |

このチュートリアルでは 2.1 で DX-COM をインストールします。DX-TRON は任意で、2.3 で扱います。

### 2.1 DX-COM のインストール

実行するコマンドは次のとおりです。

```bash
cd <DX_ALL_SUITE_DIR>
./dx-compiler/install.sh --target=dx_com
```

`run-jupyter-lab.sh` はチュートリアルの Python 環境を有効化せずに JupyterLab を起動するため、このコマンドはノートブックのセルで実行してもターミナルで実行してもシステムの `python3` を使います。インストーラーは `dx-compiler/venv-dx-compiler-local` に独自の環境を作成し、Jupyter の環境には手を加えません。

*どこで*実行するかを決める唯一の要素は `sudo` です。インストーラーは `python3-dev` と `python3-venv` のために常に `sudo apt-get` を呼び出しますが、ノートブックのセルはパスワード入力に応答できません。そのため、以下の 3 つのセルは次のことを行います。

1. **確認**: `dxcom` がすでにインストールされているか、この環境で `sudo` がパスワードなしで動作するかを確認します。
2. **インストール**: 上記のコマンドを実行します(確認セルが実行してよいと示した場合のみ実行してください)。
3. **検証**: `dxcom` が存在することを確認します。

ダウンロードは数百 MB で、2〜10 分かかります。出力の最後に緑色の `[HINT] dx_com installation completed!` ブロックが表示されます。参考のために下に掲載しています。

In [ ]:
# 1. Check: is DX-COM installed, and can this notebook run the installer?
if DXCOM_PATH.is_file():
    print(f"DX-COM is already installed: {DXCOM_PATH}")
    print("The install cell will skip itself; continue with the verify cell.")
elif subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    print("Passwordless sudo is available. Run the next cell to install DX-COM here.")
else:
    print("sudo needs a password on this host, so the installer cannot run in a cell.")
    print("Open File > New > Terminal, run the two commands below, then run the verify cell:\n")
    print(f'cd {shlex.quote(str(DX_ALL_SUITE_DIR))}')
    print("./dx-compiler/install.sh --target=dx_com")


In [ ]:
# 2. Install DX-COM (same command as in a terminal). Skipped when dxcom is already installed.
if DXCOM_PATH.is_file():
    print("DX-COM is already installed, skipping:", DXCOM_PATH)
else:
    !cd "{DX_ALL_SUITE_DIR}" && ./dx-compiler/install.sh --target=dx_com


インストールが正常に完了すると、インストーラーは次のヒントブロックを表示します。作成された仮想環境 `dx-compiler/venv-dx-compiler-local` は Jupyter の環境とは分離されています。2.2 のセルは各 `!` 行のシェル内で `source venv-dx-compiler-local/bin/activate` を実行して有効化するため、Jupyter の環境は変更されません。

<pre style="color: green">
[HINT] ==================================================================== 
[HINT]   dx_com installation completed! 
[HINT]  
[HINT]   To use dx_com, activate the virtual environment first: 
[HINT]     $ source <path_to_dx_all_suite>/dx-compiler/venv-dx-compiler-local/bin/activate 
[HINT]  
[HINT]   Then you can run dxcom: 
[HINT]     $ dxcom -h 
[HINT]  
[HINT] ==================================================================== 
</pre>

次に、DX-Compiler によってインストールされたコンポーネントを確認します。

In [ ]:
# 3. Verify: stop here with the fix if dxcom is still missing.
paths = setup_tutorial(needs=["dx_com"], quiet=True)
paths.require("dx_com")
print("DX-COM is ready:", DXCOM_PATH)


In [ ]:
!tree -L 2 "{DX_COMPILER_DIR}"

では、dx_com 配下のファイルとフォルダーを見てみましょう。

In [ ]:
!tree -L 1 "{DX_COMPILER_DIR}"/dx_com

# There are three ONNX files as sample models
!tree -L 2 "{DX_COMPILER_DIR}"/dx_com/sample_models

### 2.2 DX-Compiler の検証

2.1 の検証セルで `dxcom` が存在することはすでに確認しました。この節では実際に実行します。まずヘルプを表示し、次にサンプルモデルを実際にコンパイルします。

In [ ]:
print(f"DX-COM workspace: {DX_COM_DIR}")
print(f"dxcom executable: {DXCOM_PATH}")


#### 2.2.1 `dxcom` のヘルプを確認する

`dxcom` はインストーラーが作成した仮想環境の中にあるため、ターミナルでは呼び出す前にその環境を有効化する必要があります。次のセルは正確に以下のコマンドを実行します。

```bash
cd <DX_ALL_SUITE_DIR>/dx-compiler
source venv-dx-compiler-local/bin/activate
dxcom -h
```

ノートブックの `!` 行は毎回新しいシェルを起動するため、3 つのコマンドを `&&` で 1 行につないでいます。有効化はそのシェルにのみ適用され、Jupyter の環境は変更されません。

In [ ]:
!cd "{DX_COMPILER_DIR}" && source venv-dx-compiler-local/bin/activate && dxcom -h


#### 2.2.2 `MobileNetV2-1.onnx` をコンパイルして `MobileNetV2-1.dxnn` を生成する

次のセルは以下のコマンドを実行します。コンパイラは各ステージを出力し、`--gen_log` によって同じ内容が `compiler.log` にも書き込まれます。数秒から 1 分程度かかります。

```bash
cd <DX_ALL_SUITE_DIR>/dx-compiler
source venv-dx-compiler-local/bin/activate
cd dx_com
dxcom -m sample_models/onnx/MobileNetV2-1.onnx \
      -c sample_models/json/MobileNetV2-1.json \
      -o output/MobileNetV2-1 \
      --gen_log
```

In [ ]:
!cd "{DX_COMPILER_DIR}" && source venv-dx-compiler-local/bin/activate && cd dx_com && \
    dxcom -m sample_models/onnx/MobileNetV2-1.onnx \
          -c sample_models/json/MobileNetV2-1.json \
          -o output/MobileNetV2-1 \
          --gen_log


`MobileNetV2-1.dxnn` ファイルが生成されたか確認します。

In [ ]:
!tree -h "{DX_COM_DIR}/sample_models" "{DX_COM_DIR}/output"


`--gen_log` オプションは、コンパイルログを `compiler.log` という名前のファイルに出力します。

保存されたログメッセージを見てみましょう。

In [ ]:
!cat "{DX_COM_DIR}"/output/MobileNetV2-1/compiler.log

### 2.3 DX-Tron

**DX-TRON** は、DEEPX ツールチェーンでコンパイルした `.dxnn` モデルファイルを探索するためのグラフィカルな可視化ツールです。 

モデル構造を読み込んで確認し、色分けされたグラフで NPU と CPU の間のワークロード分配を確認できます。 

DX-TRON を使うと、モデルの実行フローをより深く理解し、全体の性能を改善できます。

> **注:** DX-TRON は引き続き SDK と共に配布されていますが、積極的なメンテナンスは行われていません。コンパイル済みモデルのレポートには、`dxcom --export_html` が生成する HTML サマリー(チュートリアル 05)を推奨します。DX-TRON は `.dxnn` ファイルを手早く視覚的に確認する用途では今も便利です。

<img src="assets/sc-dxtron.png" style="max-width: 600px;">

**主な機能:**
- **.dxnn ファイル対応**: DEEPX ツールチェーンでコンパイルしたモデルファイルを読み込んで可視化します。
- **ワークロードの視覚的表現**: ワークロードの実行を色分けして表示します。
- 赤: NPU で実行される演算
- 青: CPU またはホストで実行される演算
- **モデルナビゲーション**: 左下の戻る矢印で、いつでもモデル概要画面に戻れます。
- **対話的なノード検査**: グラフ内のノードをダブルクリックすると、関連する演算の詳細情報を表示できます。
- 参考: DX-TRON は DXNN をサポートするために [netron](https://netron.app/) をベースに開発されました。

#### 2.3.1 DX-TRON のインストール

DX-TRON は Debian パッケージとして配布されるため、インストーラーは `sudo apt-get` を使用します。

```bash
cd <DX_ALL_SUITE_DIR>
./dx-compiler/install.sh --target=dx_tron
```

2.1 と同様に 3 つのセルが続きます。**確認**(`dxtron` がインストールされているか、`sudo` がパスワードなしで動作するか)、**インストール**(上記のコマンド。確認セルが実行してよいと示した場合のみ実行するか、ターミナルで実行してください)、**検証**です。

In [ ]:
# 1. Check: is DX-TRON installed, and can this notebook run the installer?
if shutil.which("dxtron"):
    print(f"DX-TRON is already installed: {shutil.which('dxtron')}")
    print("The install cell will skip itself; continue with the verify cell.")
elif subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    print("Passwordless sudo is available. Run the next cell to install DX-TRON here.")
else:
    print("sudo needs a password on this host, so the installer cannot run in a cell.")
    print("Open File > New > Terminal, run the two commands below, then run the verify cell:\n")
    print(f'cd {shlex.quote(str(DX_ALL_SUITE_DIR))}')
    print("./dx-compiler/install.sh --target=dx_tron")


In [ ]:
# 2. Install DX-TRON (same command as in a terminal). Skipped when dxtron is already installed.
if shutil.which("dxtron"):
    print("DX-TRON is already installed, skipping:", shutil.which("dxtron"))
else:
    !cd "{DX_ALL_SUITE_DIR}" && ./dx-compiler/install.sh --target=dx_tron


In [ ]:
# 3. Verify: stop here with the fix if dxtron is still missing.
paths = setup_tutorial(needs=["dx_tron"], quiet=True)
paths.require("dx_tron")
print("DX-TRON is ready:", shutil.which("dxtron"))


#### 2.3.2 DX-TRON の実行

次のセルは、2.2.2 でコンパイルした MobileNetV2 モデルを DX-TRON で開きます。ウィンドウにはデスクトップセッションが必要で、開いている間はセルが実行中のままになるため、セルの先頭行で `RUN_DXTRON = True` にしたときだけ実行されます。既定の `False` ではコマンドを表示するだけです。3.4.4 の 2 つ目の DX-TRON セルも同じフラグを使います。
> **注:** 上の停止ボタン('■')をクリックすると `dxtron` を終了できます!

In [ ]:
RUN_DXTRON = False   # True opens the DX-TRON window (desktop session only); the cell stays busy until the window is closed

if RUN_DXTRON:
    !dxtron "{DX_COM_DIR}"/output/MobileNetV2-1/MobileNetV2-1.dxnn
else:
    print("Skipped: set RUN_DXTRON = True to open the model in DX-TRON. Same command in a terminal:")
    print(f'dxtron "{DX_COM_DIR}/output/MobileNetV2-1/MobileNetV2-1.dxnn"')


## 3. DX-Runtime のインストール
詳細は [DX-All Suite インストールガイド](https://github.com/DEEPX-AI/dx-all-suite/blob/main/docs/source/02_Setting_Up_Environment.md)を参照してください。

### インストール前のプラットフォーム別の注意 (任意)

**Orange Pi 5 Plus のみ**
 - Orange Pi の公式イメージを使用している場合、カーネルヘッダーがインストールされていないことがあります。NPU ドライバーのインストールにはカーネルヘッダーが必要です。
 - [こちら](../../docs/orangepi5p.md)にリンクされたドキュメントを参照してください。

**Raspberry Pi 5 のみ**
 - PCIe は既定で Gen2 に設定されています。帯域幅を増やすために Gen3 に設定できます。 
 - [こちら](../../docs/raspberrypi5.md)にリンクされたドキュメントを参照してください。 

DX-Runtime 環境には各モジュールのソースコードが含まれています。リポジトリは `./dx-runtime` 配下の Git サブモジュール(`dx_rt_npu_linux_driver`、`dx_fw`、`dx_rt`、`dx_app`、`dx_stream`)として管理されています。

DX-Runtime インストールスクリプトのすべてのオプションを見てみましょう。

In [ ]:
!bash "{DX_RUNTIME_DIR}"/install.sh --help

### 3.1 DX-Runtime のインストール

後のチュートリアルで DX-APP と DX-STREAM を使用するため、`--all` ですべてをインストールします。

```bash
cd <DX_ALL_SUITE_DIR>
./dx-runtime/install.sh --all
```

コアランタイム(NPU ドライバー、ファームウェア、DX-RT)だけが必要な場合は、`--all` の代わりに `--runtime-only` を使用してください。

インストーラーは全体を通して `sudo` を使用します。NPU カーネルドライバーのビルドとロード、Debian パッケージのインストール、`dxrt.service` デーモンの登録を行います。質問はしません。2.1 と同様に 3 つのセルが続きます。**確認**(インストール済みか、`sudo` がパスワードなしで動作するか)、**インストール**(上記のコマンド。確認セルが実行してよいと示した場合のみ実行するか、ターミナルで実行してください)、**検証**です。10〜30 分かかり、大部分は DX-RT、DX-APP、DX-STREAM のビルド時間です。

NPU ドライバーのインストール後は**再起動**が必要で、ファームウェア更新後はインストーラーが完全な電源オフを推奨します。再起動後、`./run-jupyter-lab.sh` を再度起動し、このノートブックを開いて最初のコードセルを実行してから 3.2 に進んでください。

In [ ]:
# 1. Check: is DX-Runtime installed, and can this notebook run the installer?
runtime_cli = shutil.which("dxcli") or shutil.which("dxrt-cli")
if runtime_cli:
    print(f"DX-Runtime is already installed: {runtime_cli}")
    print("The install cell will skip itself; continue with the verify cell.")
elif subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    print("Passwordless sudo is available. Run the next cell to install DX-Runtime here.")
else:
    print("sudo needs a password on this host, so the installer cannot run in a cell.")
    print("Open File > New > Terminal, run the two commands below, reboot, then run the verify cell:\n")
    print(f'cd {shlex.quote(str(DX_ALL_SUITE_DIR))}')
    print("./dx-runtime/install.sh --all")


In [ ]:
# 2. Install DX-Runtime (same command as in a terminal). Skipped when DX-RT is already installed. Reboot afterwards.
if shutil.which("dxcli") or shutil.which("dxrt-cli"):
    print("DX-Runtime is already installed, skipping:", shutil.which("dxcli") or shutil.which("dxrt-cli"))
else:
    !cd "{DX_ALL_SUITE_DIR}" && ./dx-runtime/install.sh --all


In [ ]:
# 3. Verify. The NPU device appears only after the reboot; the kernel driver and service are checked in 3.2.
paths = setup_tutorial(needs=["dx_rt"], quiet=True)
if paths.ok:
    print("DX-Runtime is installed. If you have not rebooted since the driver installation, reboot now, "
          "start ./run-jupyter-lab.sh again, and continue with 3.2.")
else:
    print("DX-Runtime is not installed yet:", paths.missing[0].detail)
    print("Complete the install step above, reboot, and run this cell again.")


### 3.2 インストールの検証

インストールディレクトリと Git ブランチは、設定セルによってすでに `dx-tutorials/config.json` に保存されています。この検証では、リポジトリ、ブランチ、サブモジュール、DX-Compiler 環境、DX-Runtime CLI、NPU デバイスノードを確認します。その後のセルでは、PCIe リンク、カーネルドライバー、サービスをより詳しく調べます。

In [ ]:
# Verify everything that Tutorial 01 installs. Anything MISSING is listed with the step that provides it.
paths = setup_tutorial(needs=["sdk_branch", "sdk_submodules", "dx_com", "dx_rt", "npu"])

print()
if paths.ok:
    print(f"Installation verified. Settings are stored in {CONFIG_PATH}")
else:
    print("Installation is incomplete. Complete the steps listed above and run this cell again.")


In [ ]:
# Check the PCIe generation and lane count of the NPU. Reading LnkCap/LnkSta needs root.
# DX-M1 performs best on PCIe Gen3 (8GT/s) x4; the host decides what is negotiated.
if subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    !sudo lspci -vvv | grep -iA 33 accelerators | grep -E "LnkCap|LnkSta"
else:
    print("sudo needs a password here. Run this in a terminal instead:")
    print("  sudo lspci -vvv | grep -iA 33 accelerators | grep -E 'LnkCap|LnkSta'")


In [ ]:
# Check NPU kernel driver status -> expected: dxrt_driver, dx_dma
!lsmod | grep dxrt

In [ ]:
# Check NPU device files -> expected: '/dev/dxrt0'
!ls /dev/dxrt*

In [ ]:
# Check NPU daemon status -> expected: dxrt.service - DX-RT Service is active (running) state
!systemctl status dxrt.service

### 3.3 推奨: 選択したモデルのみダウンロードする

次のコードセルは `SELECTED_MODELS` を許可リストとして使い、それらのモデルだけをダウンロードします。これがこのチュートリアルで推奨する既定の方法です。別のモデルセットが必要な場合は、セルを実行する前にリストを編集してください。

> **警告 — 必要な場合を除き、全件ダウンロードのコマンドは使用しないでください。**  
> `!cd $DX_ALL_SUITE_DIR/dx-runtime/dx_app && bash setup.sh <<< ""` を実行すると、対話型プロンプトに空の回答が与えられます。空の回答はすべてのカテゴリとすべてのモデルを選択するため、352 個のモデル(`dx_app` v3.2.2 のマニフェスト)がすべてダウンロードされます。全セットには大量のネットワークトラフィックと約 29 GB のストレージが必要です。

推奨セルは `SELECTED_MODELS` を `setup.sh --models` に渡し、`--no-force` を使って既存のモデルファイルを再ダウンロードしないようにします。リストされている 24 個のモデルはチュートリアル 02〜24 で使用するもので、ダウンロードにはネットワークによって数分かかります。以降のチュートリアルは必要なモデルだけを指定して同じ `setup.sh --models ... --no-force` を呼ぶため、このセルを一度実行しておけば、それらのダウンロードセルはすべてスキップされ数秒で終わります。

In [ ]:
# Download all models
#!cd "{DX_ALL_SUITE_DIR}"/dx-runtime/dx_app && bash setup.sh <<< ""

In [ ]:
# Download the *.dxnn models used by this tutorial and by the later tutorials

SELECTED_MODELS = shlex.join([
    "sfa3d_608x608",
    "deepmar_resnet50",
    "ResNet50",
    "depth_anything_v2_vits-1",
    "ViT-B-32-256-datacomp_s34b_b86k-1",
    "3ddfa_v2_mobilnet0.5_120x120",
    "SCRFD500M",
    "mediapipe-hand-detector-1",
    "HandLandmarkLite_1",
    "DnCNN_15",
    "zero_dce_pp-1",
    "yolo26s-seg",
    "superpoint-1",
    "yolo26s-obb",
    "yolo26s",
    "YOLOV12N_PPU",
    "yolopv2-1",
    "yolo26s-pose",
    "pidnet-s-1",
    "RealESRGAN_x4-1",
    "YoloV7",
    "YoloV7_PPU",
    "yolov8s_pose",
    "yolov8n_seg",
])

MODEL_MANIFEST_PATH = DX_APP_DIR / "scripts" / "modelzoo_manifest.json"

# Check the downloadable model list
#!cat "{MODEL_MANIFEST_PATH}" | grep name

# Download selected models only
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force

In [ ]:
# Let's check what models were downloaded in the workspace directory below.
!tree "{DX_WORKSPACE_DIR}"/res/models

# Let's check what videos were downloaded in the workspace directory below.
!tree "{DX_WORKSPACE_DIR}"/res/videos

### 3.4 DX-RT が提供する便利なツール

DX-RT がサポートする CLI ツールを見てみましょう。

In [ ]:
!ls -l /usr/local/bin | grep -E "dx|model"

**現在の CLI 名と後方互換の名前**

DX-RT は現在の CLI バイナリを `/usr/local/bin` にインストールします。以前のコマンド名は、現在のバイナリへのシンボリックリンクとして引き続き利用できます。

| 現在の名前 | 以前の互換名 | 実装 |
|---|---|---|
| `dxcli` | `dxrt-cli` | `dxrt-cli -> dxcli` |
| `dxrun` | `run_model` | `run_model -> dxrun` |
| `dxparse` | `parse_model` | `parse_model -> dxparse` |

したがって、どちらの名前でも同じバイナリが実行され、同じオプションを受け付けます。ヘルプには呼び出しに使った名前が表示されることがあります。`dxbenchmark`、`dxtop`、`dxrtd` は既存の名前のままで、独立したバイナリとしてインストールされます。

#### 3.4.1 dxbenchmark

`dxbenchmark` は、コンパイル済みの .dxnn モデルを実行して機能をテストし、性能を測定する CLI ツールです。

In [ ]:
# Check dxbenchmark tool
!dxbenchmark -h

In [ ]:
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"
WORKSPACE_DIR.mkdir(exist_ok=True)   # dxbenchmark writes its reports into the current directory

# Benchmark every downloaded model (24 models x 5 loops: expect a few minutes).
!cd "{WORKSPACE_DIR}" && dxbenchmark --dir "{MODEL_DIR}" -l 5


In [ ]:
# Three report formats are generated: csv, json, and html.
!ls "{WORKSPACE_DIR}"/DXBENCHMARK*


Web ブラウザーを開いて、`dxbenchmark` の結果を HTML 形式で確認します。

In [ ]:
import base64
from pathlib import Path

from IPython.display import Javascript, display

report_directories = {
    WORKSPACE_DIR.resolve(),
    Path.cwd().resolve(),
}
benchmark_reports = [
    report
    for directory in report_directories
    for report in directory.glob("DXBENCHMARK_*.html")
]
if not benchmark_reports:
    raise FileNotFoundError("No DXBENCHMARK_*.html report was found. Run dxbenchmark first.")

latest_report = max(benchmark_reports, key=lambda path: path.stat().st_mtime)
encoded_report = base64.b64encode(latest_report.read_bytes()).decode("ascii")

display(Javascript(f'''(() => {{
    const binary = atob("{encoded_report}");
    const bytes = Uint8Array.from(binary, character => character.charCodeAt(0));
    const blob = new Blob([bytes], {{type: "text/html"}});
    const reportUrl = URL.createObjectURL(blob);
    window.open(reportUrl, "_blank");
    setTimeout(() => URL.revokeObjectURL(reportUrl), 60000);
}})();'''))
print(f"Opened benchmark report: {latest_report}")
print("If no new tab appeared, your browser blocked the popup. Open the file above directly in a browser.")

#### 3.4.2 dxtop

`dxtop` は、使用率、温度、メモリなどの DEEPX NPU メトリクスをリアルタイムで監視する htop 風のツールです。

> dxtop の出力形式は Jupyter Notebook のコードセルでは正しく表示されません。代わりに**別のターミナルを開いて** dxtop コマンドを実行してください。 </br>
> 別のターミナルを開くには: `File > New > Terminal`
> 
> ![](assets/open-terminal.png)

> <img src="assets/sc-dxtop.png" style="max-width: 600px;">

In [ ]:
# Check dxtop tool - Open a separate terminal and run 'dxtop'
!dxtop -h
#!dxtop

#### 3.4.3 dxcli

`dxcli`(互換性のため `dxrt-cli` としても利用可能)は、DEEPX DX-RT デバイスの照会と監視、および NPU ファームウェアの管理を行います。

In [ ]:
# Check dxcli tool
!dxcli -h

In [ ]:
# Check the NPU status
!dxcli -s

In [ ]:
# Check the NPU F/W version
!dxcli -i | grep FW

#### 任意: NPU ファームウェアの書き込み

DX-Runtime インストーラーはこの SDK バージョンに対応するファームウェアをすでに書き込んでいるため、この手順は通常**不要**です。DEEPX サポートから再書き込みを求められた場合や、別の SDK ブランチに切り替えた後にのみ使用してください。次のセルで `FLASH_FIRMWARE = True` に設定すると有効になります。`False` の場合、セルは実行予定の内容を表示するだけです。

コマンドは次のとおりです。

```bash
sudo systemctl stop dxrt.service
dxcli -u <DX_ALL_SUITE_DIR>/dx-runtime/dx_fw/m1/latest/mdot2/fw.bin   # DX-H1 は h1/fw.bin
sleep 5
sudo systemctl start dxrt.service
```

In [ ]:
####################################################################
FLASH_FIRMWARE = False   # set to True only when a re-flash is really required
####################################################################

FIRMWARE_PATH = DX_RUNTIME_DIR / "dx_fw" / "m1" / "latest" / "mdot2" / "fw.bin"
# For DX-H1 hardware use: DX_RUNTIME_DIR / "dx_fw" / "m1" / "latest" / "h1" / "fw.bin"

if not FLASH_FIRMWARE:
    print(f"Skipped. Firmware image that would be written: {FIRMWARE_PATH}")
elif not FIRMWARE_PATH.is_file():
    raise FileNotFoundError(f"Firmware image was not found: {FIRMWARE_PATH}")
else:
    !sudo systemctl stop dxrt.service
    !dxcli -u "{FIRMWARE_PATH}"
    !sleep 5
    !sudo systemctl start dxrt.service


In [ ]:
# Check the NPU F/W version
!dxcli -i | grep FW

#### 3.4.4 dxrun

`dxrun`(互換性のため `run_model` としても利用可能)は、コンパイル済みの .dxnn モデルを実行して機能をテストし、性能を測定する CLI ツールです。

In [ ]:
# Check dxrun tool
!dxrun -h

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"
!dxrun -m "{MODEL_FILE}"

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"
RUN_DXTRON = False   # True opens the DX-TRON window (desktop session only); the cell stays busy until the window is closed

if RUN_DXTRON:
    !dxtron "{MODEL_FILE}"
else:
    print("Skipped: set RUN_DXTRON = True to open the model in DX-TRON. Same command in a terminal:")
    print(f'dxtron "{MODEL_FILE}"')


`yolo26-s_640x640.dxnn` は 2 つの部分(NPU と CPU)で構成されています。

 ![](assets/sc-yolo26s.png)

NPU がサポートしない演算子は、CPU オフロードによって ONNX Runtime を使い CPU で処理されます。 

`dxrun` に `--use-ort` オプションを追加すると NPU と CPU の両方の部分が実行され、.dxnn AI パイプライン全体の性能を測定できます。

`--use-ort` オプションを省略すると、性能測定は NPU 部分のみに限定されます。

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

!echo "============================= NO ORT ================================="
!dxrun -m "{MODEL_FILE}" -l 200 | grep FPS
!echo "======================================================================"
!echo ""
!echo "============================= USE ORT ================================"
!dxrun -m "{MODEL_FILE}" -l 200 --use-ort | grep FPS
!echo "======================================================================"

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

# -t 5: For 5 seconds, -v: Shows NPU Processing Time and Latency
!dxrun -m "{MODEL_FILE}" --use-ort -t 5 -v

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

# --profiler writes profiler.json into the current directory, so run it inside the tutorial workspace.
WORKSPACE_DIR.mkdir(exist_ok=True)
!cd "{WORKSPACE_DIR}" && dxrun -m "{MODEL_FILE}" --use-ort -t 5 -v --profiler


#### 3.4.5 dxparse

`dxparse`(互換性のため `parse_model` としても利用可能)は、コンパイル済みの .dxnn モデルを読み込み、その構造、入出力、メタデータを表示するコマンドラインツールです。

In [ ]:
!dxparse -h

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"
!dxparse -m "{MODEL_FILE}" -v

## 4. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| インストールセルが `sudo` で止まる | `sudo: a terminal is required to read the password` | パスワードなしの sudo が設定されておらず、インストーラーがセル内でパスワードを要求できない | 表示されたコマンドをターミナル(File > New > Terminal)で実行し、その後検証セルを実行する |
| クローンセルが失敗する | `fatal: destination path ... already exists and is not an empty directory` | 対象の場所に git チェックアウトではないファイルが存在する | ディレクトリを移動するか 1.1 で別の場所を選び、セルを再実行する |
| `dxcom` が MISSING のまま | ステータス表に `[MISSING] dx_com` | DX-COM インストーラーが最後まで完了していない | 上にスクロールしてインストーラーの出力を確認する。`[HINT]` ブロックが必要。2.1 を再実行する |
| NPU デバイスがない | `[MISSING] npu` または `ls: cannot access '/dev/dxrt*'` | カーネルドライバーがロードされていない。多くは 3.1 の後にホストを再起動していない | 再起動する。次に `lsmod \| grep dxrt` を確認し、それでもデバイスがなければホストの電源を完全に切ってから入れ直す |
| `dxrt.service` が動作していない | `systemctl status dxrt.service` に `Active: failed` | ドライバーまたはファームウェアの不一致 | `sudo systemctl restart dxrt.service`。再び失敗する場合はターミナルで `./dx-runtime/install.sh --runtime-only` を実行する |
| モデルがダウンロードされない | `setup.sh` が不明なモデル名を報告する | 名前が Model Zoo のマニフェストと異なる(大文字小文字を区別) | `dx_app` 内で `grep name scripts/modelzoo_manifest.json` を実行して正確な名前を確認する |

## 5. まとめ

| 項目 | DX-Compiler | DX-Runtime |
|---|---|---|
| 役割 | モデルのコンパイル | モデル推論の実行 |
| 入力 | ONNX | DXNN |
| 出力 | `.dxnn` | 推論結果 |
| システム | x86_64 のみ | x86_64、aarch64 |
| インストール方法 | `./dx-compiler/install.sh --target=dx_com` | `./dx-runtime/install.sh --all` |
| 主な CLI | `dxcom`(`venv-dx-compiler-local` 内) | `dxcli`、`dxrun`、`dxparse`、`dxbenchmark`、`dxtop` |

### 5.1 完了チェックリスト

- [ ] `config.json` が DX-All Suite ディレクトリを指し、1.3 のステータスセルですべての項目が `OK` と表示される
- [ ] `dxcom -h` が実行でき、`MobileNetV2-1.dxnn` がコンパイルされた (2.2)
- [ ] `dxtron` で `.dxnn` ファイルを開ける (2.3、任意)
- [ ] `/dev/dxrt0` が存在し、`dxrt.service` がアクティブで、`dxcli -s` に NPU が表示される (3.2)
- [ ] 選択したモデルが共有ワークスペースにダウンロードされた (3.3)
- [ ] `dxrun` と `dxbenchmark` で NPU 上でモデルを実行した (3.4)

> **次へ:** チュートリアル 02 に進み、ここでダウンロードしたモデルで DX-APP のサンプル(分類、検出、姿勢推定、セグメンテーション)を実行してみましょう。